#  DenseNet-121 — HAM10000 Balancing Strategy Comparison
### AI-Powered Multimodal Skin Disease Detection — HAM10000
**Student:** Mohamed Alhusein Abdalla | **Supervisor:** Dr. Gehad Ismail Sayed, BUE

### Balancing Strategies:
| Strategy | Description |
|---|---|
| S1 — Weighted Loss Only | Original imbalanced data + class-weighted CE |
| S2 — Aug-Based Oversampling | Oversample minorities; each copy gets different transforms |
| S3 — Hybrid | Aug-oversampling + Weighted CE |

### Key Setup:
-  Offline hair removal (Black-Hat, threshold=15) + CLAHE
-  224×224 + ImageNet norm
-  MixUp (alpha=0.2) + ColorJitter
-  Dropout 0.5 + BatchNorm in head
-  AdamW | Frozen backbone (feature extraction only)
-  Label Smoothing CE (0.1) | Macro F1 primary metric | Early stop patience=7

---


## Step 0 — Configuration

In [1]:
import os

# ── Kaggle ─────────────────────────────────────────────────────────────────
KAGGLE_USERNAME = 'mhmdabdalla'
KAGGLE_KEY      = 'KGAT_7f704b345481aaeddc1aca0cefab4295'

# ── Paths ──────────────────────────────────────────────────────────────────
HAM_DIR      = '/workspace/HAM10000'
IMG_DIR_1    = '/workspace/HAM10000/HAM10000_images_part_1'
IMG_DIR_2    = '/workspace/HAM10000/HAM10000_images_part_2'
CSV_PATH     = '/workspace/HAM10000/HAM10000_metadata.csv'
CLEANED_DIR  = '/workspace/HAM10000/cleaned_images'
SAVE_DIR     = '/workspace/checkpoints'
RESULTS_DIR  = '/workspace/results'

# ── Hyperparameters ────────────────────────────────────────────────────────
NUM_CLASSES  = 7
BATCH_SIZE   = 64
NUM_EPOCHS   = 30
EARLY_STOP   = 7
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.5
MIXUP_ALPHA  = 0.2
MAX_SAMPLES  = 1000
RANDOM_SEED  = 42
CLASS_NAMES  = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

for d in [HAM_DIR, CLEANED_DIR, SAVE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Configuration done!')
print(f'Epochs: {NUM_EPOCHS} | Early stop: {EARLY_STOP} | Batch: {BATCH_SIZE}')
print(f'LR={LR_HEAD} | WD={WEIGHT_DECAY} | Dropout={DROPOUT} | MixUp={MIXUP_ALPHA}')
print(f'MAX_SAMPLES per class: {MAX_SAMPLES} | Frozen backbone (feature extraction only)')

Configuration done!
Epochs: 30 | Early stop: 7 | Batch: 64
LR=0.001 | WD=0.0001 | Dropout=0.5 | MixUp=0.2
MAX_SAMPLES per class: 1000 | Frozen backbone (feature extraction only)


##  Step 1 — Install Missing Libraries

In [2]:
import sys
!{sys.executable} -m pip install pandas scikit-learn opencv-python-headless seaborn timm kaggle scikit-image torchmetrics -q
print('Libraries ready!')

Libraries ready!


## Step 2 — Imports

In [3]:
import os, sys, random, warnings, json, zipfile, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from sklearn.utils import resample

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, f1_score)

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA H200
VRAM: 150.1 GB


## Step 3 — Kaggle Setup & Download

In [4]:
import subprocess
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured!')

Kaggle credentials configured!


In [5]:
if not os.path.exists(CSV_PATH):
    print('Downloading HAM10000...')
    result = subprocess.run(
        [sys.executable, '-m', 'kaggle', 'datasets', 'download',
         '-d', 'kmader/skin-cancer-mnist-ham10000', '-p', HAM_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
    zip_path = os.path.join(HAM_DIR, 'skin-cancer-mnist-ham10000.zip')
    if os.path.exists(zip_path):
        print('Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(HAM_DIR)
        os.remove(zip_path)
        print('Zip deleted!')
    else:
        print('ERROR: zip not found. Check Kaggle API key.')
else:
    print('Dataset already exists! Skipping.')

print('CSV:', os.path.exists(CSV_PATH))
print('Part1:', os.path.exists(IMG_DIR_1), '| Part2:', os.path.exists(IMG_DIR_2))

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0


Extracting...
Zip deleted!
CSV: True
Part1: True | Part2: True


## Step 4 — Load Images & CSV

In [6]:
image_paths = {}
for folder in [IMG_DIR_1, IMG_DIR_2]:
    if os.path.exists(folder):
        for fname in os.listdir(folder):
            if fname.lower().endswith('.jpg'):
                image_paths[fname.replace('.jpg','').replace('.JPG','')] = os.path.join(folder, fname)

print(f'Images found: {len(image_paths)}')
if len(image_paths) < 100:
    print('Searching deeper...')
    for root, dirs, files in os.walk(HAM_DIR):
        for fname in files:
            if fname.lower().endswith('.jpg'):
                image_paths[fname.replace('.jpg','').replace('.JPG','')] = os.path.join(root, fname)
    print(f'Found after search: {len(image_paths)}')

Images found: 10015


In [7]:
df = pd.read_csv(CSV_PATH)
df['filepath'] = df['image_id'].map(image_paths)
df = df.dropna(subset=['filepath']).reset_index(drop=True)
le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])
print(f'Dataset: {len(df)} rows | Classes: {list(le.classes_)}')
print(df['dx'].value_counts())

Dataset: 10015 rows | Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


## Step 4b — EDA
*Class distribution, sample images, pixel statistics.*

In [8]:
# ── Class distribution bar chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['dx'].value_counts()
colors = sns.color_palette('Set2', len(counts))
bars   = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Class Distribution — HAM10000', fontsize=13)
axes[0].set_xlabel('Lesion Class'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution (%) — HAM10000', fontsize=13)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Imbalance ratio (nv / rarest):', round(counts.max() / counts.min(), 1))

Imbalance ratio (nv / rarest): 58.3


In [9]:
# ── Sample images per class ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
fig.suptitle('Sample Images per Class — HAM10000', fontsize=13)

for col, cls in enumerate(CLASS_NAMES):
    for row in range(2):
        sample = df[df['dx'] == cls].sample(1, random_state=RANDOM_SEED + row).iloc[0]
        img    = Image.open(sample['filepath']).convert('RGB').resize((128, 128))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(cls, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample images saved.')

Sample images saved.


In [10]:
# ── Metadata distributions ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Age distribution
df['age'].dropna().hist(ax=axes[0], bins=20, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('Age Distribution'); axes[0].set_xlabel('Age'); axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3)

# Sex distribution
sex_counts = df['sex'].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values, color=['#2196F3','#FF9800','#9E9E9E'],
            edgecolor='black', linewidth=0.5)
axes[1].set_title('Sex Distribution'); axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3, axis='y')

# Localization distribution
loc_counts = df['localization'].value_counts()
axes[2].barh(loc_counts.index, loc_counts.values, color='#4CAF50', edgecolor='black', linewidth=0.5)
axes[2].set_title('Localization Distribution'); axes[2].set_xlabel('Count')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/eda_metadata.png', dpi=150, bbox_inches='tight')
plt.show()
print('Metadata EDA saved.')

Metadata EDA saved.


## Step 5 — Split FIRST, Then Oversample Train Only
*Critical order: split before oversampling prevents data leakage.*
*Test and validation sets remain original — clean and comparable to literature.*

In [11]:
# ── Step 5a: Stratified split on ORIGINAL data ───────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=RANDOM_SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_SEED, stratify=temp_df['label']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print('BEFORE oversampling:')
print(f'  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('  Train class distribution:')
print(train_df['dx'].value_counts().to_string())

BEFORE oversampling:
  Train: 7010 | Val: 1502 | Test: 1503
  Train class distribution:
dx
nv       4693
mel       779
bkl       769
bcc       360
akiec     229
vasc       99
df         81


In [12]:
# ── Step 5b: Oversample ONLY train set ───────────────────────────────────
# Each class resampled to MAX_SAMPLES (with replacement)
# Combined with augmentation in training → each draw gets different transforms
# Val and test remain untouched — no leakage possible

train_balanced = pd.concat([
    resample(
        train_df[train_df['dx'] == cls],
        replace=True,
        n_samples=MAX_SAMPLES,
        random_state=RANDOM_SEED
    )
    for cls in train_df['dx'].unique()
])

train_balanced = train_balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('AFTER oversampling (train only):')
print(f'  Train: {len(train_balanced)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('  Balanced train class distribution:')
print(train_balanced['dx'].value_counts().to_string())
print()
print(f'Val and Test are ORIGINAL — no leakage!')

AFTER oversampling (train only):
  Train: 7000 | Val: 1502 | Test: 1503
  Balanced train class distribution:
dx
bcc      1000
df       1000
nv       1000
akiec    1000
bkl      1000
vasc     1000
mel      1000

Val and Test are ORIGINAL — no leakage!


In [13]:
for dframe in [train_df, train_balanced, val_df, test_df]:
    if 'filepath_clean' not in dframe.columns:
        dframe['filepath_clean'] = dframe['filepath']

## Step 6 — Hair Removal + CLAHE (One-Time, threshold=15)
*threshold=15 preserves more fine texture. CLAHE enhances local contrast for DenseNet feature propagation.*

In [14]:
def remove_hair(img_array, threshold=15):
    gray    = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    bh      = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(bh, threshold, 255, cv2.THRESH_BINARY)
    return cv2.inpaint(img_array, mask, inpaintRadius=6, flags=cv2.INPAINT_TELEA)

def apply_clahe(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)

existing = len([f for f in os.listdir(CLEANED_DIR) if f.endswith('.jpg')]) if os.path.exists(CLEANED_DIR) else 0
total    = len(image_paths)

if existing >= int(total * 0.98):
    print(f'Hair removal + CLAHE already done! ({existing}/{total})')
else:
    print(f'Running hair removal + CLAHE on {total} images (threshold=15)...')
    failed = 0
    for idx, (img_id, path) in enumerate(image_paths.items()):
        out_path = os.path.join(CLEANED_DIR, f'{img_id}.jpg')
        if os.path.exists(out_path):
            continue
        try:
            img         = cv2.imread(path)
            img_rgb     = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            cleaned     = remove_hair(img_rgb, threshold=15)
            cleaned_bgr = apply_clahe(cv2.cvtColor(cleaned, cv2.COLOR_RGB2BGR))
            cv2.imwrite(out_path, cleaned_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        except Exception:
            failed += 1
            import shutil; shutil.copy(path, out_path)
        if (idx + 1) % 2000 == 0:
            print(f'  {idx+1}/{total}...')
    print(f'Done! {total-failed} cleaned+CLAHE | {failed} copied as-is')

cleaned_paths = {f.replace('.jpg',''): os.path.join(CLEANED_DIR, f)
                 for f in os.listdir(CLEANED_DIR) if f.endswith('.jpg')}
print(f'Cleaned images: {len(cleaned_paths)}')

for dframe in [train_df, train_balanced, val_df, test_df]:
    dframe['filepath_clean'] = dframe['image_id'].map(cleaned_paths).fillna(dframe['filepath'])
print('Cleaned paths added!')

Running hair removal + CLAHE on 10015 images (threshold=15)...
  2000/10015...
  4000/10015...
  6000/10015...
  8000/10015...
  10000/10015...
Done! 10015 cleaned+CLAHE | 0 copied as-is
Cleaned images: 10015
Cleaned paths added!


## Step 7 — Transforms

In [15]:
IMGNET_MEAN = [0.485, 0.456, 0.406]
IMGNET_STD  = [0.229, 0.224, 0.225]

# Strong aug — used for oversampled train set (S2, S3)
den_train_strong = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMGNET_MEAN, IMGNET_STD),
])

# Mild aug — used for original imbalanced train set (S1)
den_train_mild = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMGNET_MEAN, IMGNET_STD),
])

# Val/Test — no aug
den_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMGNET_MEAN, IMGNET_STD),
])

print('Transforms defined!')
print('  Strong: 224x224 + flip + rotate + ColorJitter + Affine')
print('  Mild  : 224x224 + flip + rotate + mild ColorJitter')
print('  Val   : 224x224 + Normalize only')

Transforms defined!
  Strong: 224x224 + flip + rotate + ColorJitter + Affine
  Mild  : 224x224 + flip + rotate + mild ColorJitter
  Val   : 224x224 + Normalize only


## Step 8 — MixUp Augmentation

In [16]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx   = torch.randperm(x.size(0), device=x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print('MixUp ready!')

MixUp ready!


## Step 9 — Dataset Class
*No WeightedRandomSampler needed — oversampling already balanced the training set.*

In [17]:
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None, use_cleaned=False):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform
        self.path_col  = 'filepath_clean' if (use_cleaned and 'filepath_clean' in dataframe.columns) else 'filepath'

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = row[self.path_col]
        try:
            image = Image.open(path).convert('RGB')
        except Exception:
            image = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(int(row['label']), dtype=torch.long)


def make_loaders(tr_tf, val_tf, train_data=None, use_cleaned=False):
    if train_data is None:
        train_data = train_df
    train_ds = SkinDataset(train_data, tr_tf,  use_cleaned)
    val_ds   = SkinDataset(val_df,    val_tf, use_cleaned)
    test_ds  = SkinDataset(test_df,   val_tf, use_cleaned)

    # multiprocessing_context='fork' required in Jupyter to avoid pickling error
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, persistent_workers=True,
                              multiprocessing_context='fork')
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True,
                              multiprocessing_context='fork')
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True,
                              multiprocessing_context='fork')
    return train_loader, val_loader, test_loader

print('Dataset class ready!')
print('Note: Using shuffle=True (not WeightedRandomSampler) — oversampling already balanced classes.')

Dataset class ready!
Note: Using shuffle=True (not WeightedRandomSampler) — oversampling already balanced classes.


## Step 10 — Loss Functions

In [18]:
class LabelSmoothingCE(nn.Module):
    """Label Smoothing Cross Entropy."""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        n  = inputs.size(1)
        lp = F.log_softmax(inputs, dim=1)
        with torch.no_grad():
            st = torch.full_like(lp, self.smoothing / (n - 1))
            st.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        return (-st * lp).sum(dim=1).mean()


from collections import Counter
counts  = Counter(train_df['label'].values)
total_n = sum(counts.values())
weights = torch.tensor(
    [total_n / (NUM_CLASSES * counts[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(DEVICE)

print('Class weights (from original train set):')
for cls, w in zip(CLASS_NAMES, weights.cpu()):
    print(f'  {cls}: {w:.4f}')

criterion_weighted = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
criterion_standard = LabelSmoothingCE(smoothing=0.1)

print('\nLoss functions ready!')
print('  S1: Weighted CE + Label Smoothing')
print('  S2: Label Smoothing CE only')
print('  S3: Weighted CE + Label Smoothing')

Class weights (from original train set):
  akiec: 4.3731
  bcc: 2.7817
  bkl: 1.3022
  df: 12.3633
  mel: 1.2855
  nv: 0.2134
  vasc: 10.1154

Loss functions ready!
  S1: Weighted CE + Label Smoothing
  S2: Label Smoothing CE only
  S3: Weighted CE + Label Smoothing


## Step 11 — Model Builder (DenseNet-121)
*Head includes BatchNorm for training stability. Backbone frozen — feature extraction only.*

In [19]:
def build_densenet():
    base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    fdim = base.classifier.in_features  # 1024
    base.classifier = nn.Identity()

    class DenseNet_HAM(nn.Module):
        def __init__(self):
            super().__init__()
            self.base = base
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.head = nn.Sequential(
                nn.BatchNorm1d(fdim),
                nn.Dropout(DROPOUT),
                nn.Linear(fdim, 512),
                nn.ReLU(),
                nn.BatchNorm1d(512),
                nn.Dropout(DROPOUT * 0.6),
                nn.Linear(512, NUM_CLASSES)
            )
        def forward(self, x):
            f = F.relu(self.base.features(x), inplace=True)
            f = self.pool(f).flatten(1)
            return self.head(f)

    return DenseNet_HAM()


def freeze_backbone(model):
    for param in model.base.parameters():
        param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Backbone frozen. Trainable params: {trainable:,}')


print('Testing model build...')
m = build_densenet().to(DEVICE)
m.train()
x = torch.randn(2, 3, 224, 224).to(DEVICE)
out = m(x)
print(f'  DenseNet-121: 224x224 -> {list(out.shape)} ✅')
del m, x, out
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Model builder ready!')

Testing model build...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:01<00:00, 22.2MB/s]


  DenseNet-121: 224x224 -> [2, 7] ✅
Model builder ready!


## Step 12 — Training Functions

In [20]:
def train_epoch(model, loader, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = mixup_criterion(criterion, out, labels_a, labels_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item()
        correct  += (out.argmax(1) == labels_a).sum().item()
        total    += labels.size(0)

    return loss_sum / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    preds_all, labels_all, probs_all = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out   = model(imgs)
            loss  = criterion(out, labels)
            probs = torch.softmax(out, dim=1)
            loss_sum += loss.item()
            p = out.argmax(1)
            correct += (p == labels).sum().item()
            total   += labels.size(0)
            preds_all.extend(p.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    acc      = correct / total
    macro_f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(np.eye(NUM_CLASSES)[labels_all],
                            np.array(probs_all),
                            multi_class='ovr', average='macro')
    except Exception:
        auc = 0.0

    return loss_sum / len(loader), acc, preds_all, labels_all, auc


print('Train/eval functions ready!')

Train/eval functions ready!


## Step 13 — Experiment Runner

In [21]:
def run_experiment(model, exp_name, tr_loader, val_loader, te_loader):
    print(f'\n{"="*70}')
    print(f'MODEL: {exp_name}')
    print(f'Frozen backbone | Head only | LR={LR_HEAD} | Epochs={NUM_EPOCHS} | Early stop={EARLY_STOP}')
    print(f'{"="*70}')

    model      = model.to(DEVICE)
    save_path  = f'{SAVE_DIR}/{exp_name}.pth'
    best_val   = 0.0
    no_improve = 0
    history    = {'tr_loss':[], 'vl_loss':[], 'tr_acc':[], 'vl_acc':[]}
    t0         = time.time()

    freeze_backbone(model)
    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )

    print(f'\n{"Ep":>4} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>8} | {"Best":>8} | Pat')
    print('-' * 68)

    for ep in range(NUM_EPOCHS):
        tl, ta       = train_epoch(model, tr_loader, optimizer)
        vl, va, _, _, _ = evaluate(model, val_loader)
        history['tr_loss'].append(tl); history['vl_loss'].append(vl)
        history['tr_acc'].append(ta);  history['vl_acc'].append(va)

        if va >= best_val:
            best_val = va; no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1

        print(f'{ep+1:>4} | {tl:>10.4f} | {ta:>9.4f} | {vl:>8.4f} | {va:>8.4f} | {best_val:>8.4f} | {no_improve}/{EARLY_STOP}')

        if no_improve >= EARLY_STOP:
            print(f'Early stopping at epoch {ep+1}')
            break

    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))

    _, test_acc, preds, labels, auc_score = evaluate(model, te_loader)
    report   = classification_report(labels, preds, target_names=CLASS_NAMES,
                                     zero_division=0, output_dict=True)
    macro_f1 = report['macro avg']['f1-score']
    elapsed  = (time.time() - t0) / 60

    print(f'\n{"="*70}')
    print(f'RESULTS: {exp_name}')
    print(f'  Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
    print(f'  Macro F1      : {macro_f1:.4f}')
    print(f'  Macro AUC-ROC : {auc_score:.4f}')
    print(f'  Best Val Acc  : {best_val:.4f}')
    print(f'  Total time    : {elapsed:.1f} min')
    print(f'{"="*70}')
    print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))

    gap = history['tr_acc'][-1] - best_val
    print(f'Overfitting gap (final train acc - best val acc): {gap:.4f}', end=' ')
    if gap > 0.15:   print('⚠ HIGH')
    elif gap > 0.08: print('~ Moderate')
    else:            print('✓ Acceptable')

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'exp': exp_name, 'test_acc': test_acc,
        'macro_f1': macro_f1, 'auc': auc_score,
        'best_val': best_val, 'history': history,
        'preds': preds, 'labels': labels
    }


def plot_learning_curves(history, title, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    eps = range(1, len(history['tr_loss']) + 1)

    ax1.plot(eps, history['tr_loss'], 'b-o', markersize=3, label='Train Loss')
    ax1.plot(eps, history['vl_loss'], 'r-o', markersize=3, label='Val Loss')
    ax1.set_title(f'{title} — Loss Curve', fontsize=12)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(eps, history['tr_acc'], 'b-o', markersize=3, label='Train Acc')
    ax2.plot(eps, history['vl_acc'], 'r-o', markersize=3, label='Val Acc')
    ax2.set_title(f'{title} — Accuracy Curve', fontsize=12)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)

    # Shade overfitting region where val > train loss
    for ax, tr_k, vl_k in [(ax1,'tr_loss','vl_loss'), (ax2,'tr_acc','vl_acc')]:
        tr_vals = history[tr_k]; vl_vals = history[vl_k]
        ax.fill_between(eps, tr_vals, vl_vals, alpha=0.08, color='red', label='Gap')

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion(labels, preds, title, save_path=None):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title(f'{title} — Raw Counts')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title(f'{title} — Normalized')
    axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print('All training functions ready!')

All training functions ready!


## Step 14a — Strategy 1: Weighted Loss Only

In [22]:
criterion = criterion_weighted
tr1, vl1, te1 = make_loaders(den_train_mild, den_val, train_data=train_df, use_cleaned=True)
r_s1 = run_experiment(build_densenet(), 'DenseNet121_S1_WeightedLossOnly', tr1, vl1, te1)
plot_learning_curves(r_s1['history'], 'S1 — Weighted Loss Only', f'{RESULTS_DIR}/den_s1_curves.png')
plot_confusion(r_s1['labels'], r_s1['preds'], 'S1 — Weighted Loss Only', f'{RESULTS_DIR}/den_s1_cm.png')
print(f'S1 DONE | Acc={r_s1["test_acc"]:.4f} | F1={r_s1["macro_f1"]:.4f} | AUC={r_s1["auc"]:.4f}')


MODEL: DenseNet121_S1_WeightedLossOnly
Frozen backbone | Head only | LR=0.001 | Epochs=30 | Early stop=7
Backbone frozen. Trainable params: 531,463

  Ep | Train Loss | Train Acc | Val Loss |  Val Acc |     Best | Pat
--------------------------------------------------------------------
   1 |     2.5764 |    0.2194 |   2.2166 |   0.3029 |   0.3029 | 0/7
   2 |     2.3605 |    0.2300 |   2.1579 |   0.3382 |   0.3382 | 0/7
   3 |     2.2783 |    0.2161 |   2.1140 |   0.3236 |   0.3382 | 1/7
   4 |     2.2683 |    0.2097 |   2.0846 |   0.3675 |   0.3675 | 0/7
   5 |     2.2317 |    0.2415 |   2.1101 |   0.4361 |   0.4361 | 0/7
   6 |     2.2198 |    0.2368 |   2.0671 |   0.3342 |   0.4361 | 1/7
   7 |     2.2712 |    0.2335 |   2.0980 |   0.3229 |   0.4361 | 2/7
   8 |     2.2095 |    0.2254 |   2.0552 |   0.3815 |   0.4361 | 3/7
   9 |     2.2236 |    0.2245 |   2.0338 |   0.3822 |   0.4361 | 4/7
  10 |     2.2361 |    0.2324 |   2.0287 |   0.3702 |   0.4361 | 5/7
  11 |     2.2091 |   

## Step 14b — Strategy 2: Augmentation-Based Oversampling

In [23]:
criterion = criterion_standard
tr2, vl2, te2 = make_loaders(den_train_strong, den_val, train_data=train_balanced, use_cleaned=True)
r_s2 = run_experiment(build_densenet(), 'DenseNet121_S2_AugOversampling', tr2, vl2, te2)
plot_learning_curves(r_s2['history'], 'S2 — Aug Oversampling', f'{RESULTS_DIR}/den_s2_curves.png')
plot_confusion(r_s2['labels'], r_s2['preds'], 'S2 — Aug Oversampling', f'{RESULTS_DIR}/den_s2_cm.png')
print(f'S2 DONE | Acc={r_s2["test_acc"]:.4f} | F1={r_s2["macro_f1"]:.4f} | AUC={r_s2["auc"]:.4f}')


MODEL: DenseNet121_S2_AugOversampling
Frozen backbone | Head only | LR=0.001 | Epochs=30 | Early stop=7
Backbone frozen. Trainable params: 531,463

  Ep | Train Loss | Train Acc | Val Loss |  Val Acc |     Best | Pat
--------------------------------------------------------------------
   1 |     1.5884 |    0.3166 |   1.3823 |   0.5573 |   0.5573 | 0/7
   2 |     1.4658 |    0.3789 |   1.2188 |   0.6398 |   0.6398 | 0/7
   3 |     1.4125 |    0.3537 |   1.1453 |   0.6778 |   0.6778 | 0/7
   4 |     1.3810 |    0.3717 |   1.2022 |   0.6338 |   0.6778 | 1/7
   5 |     1.3722 |    0.3851 |   1.2330 |   0.6258 |   0.6778 | 2/7
   6 |     1.3733 |    0.3990 |   1.2033 |   0.6458 |   0.6778 | 3/7
   7 |     1.3325 |    0.4359 |   1.1861 |   0.6531 |   0.6778 | 4/7
   8 |     1.3454 |    0.4177 |   1.1879 |   0.6465 |   0.6778 | 5/7
   9 |     1.3203 |    0.3846 |   1.1260 |   0.6851 |   0.6851 | 0/7
  10 |     1.3387 |    0.4054 |   1.1591 |   0.6771 |   0.6851 | 1/7
  11 |     1.2936 |    

## Step 14c — Strategy 3: Hybrid (Aug-Oversampling + Weighted CE)

In [24]:
criterion = criterion_weighted
tr3, vl3, te3 = make_loaders(den_train_strong, den_val, train_data=train_balanced, use_cleaned=True)
r_s3 = run_experiment(build_densenet(), 'DenseNet121_S3_Hybrid', tr3, vl3, te3)
plot_learning_curves(r_s3['history'], 'S3 — Hybrid', f'{RESULTS_DIR}/den_s3_curves.png')
plot_confusion(r_s3['labels'], r_s3['preds'], 'S3 — Hybrid', f'{RESULTS_DIR}/den_s3_cm.png')
print(f'S3 DONE | Acc={r_s3["test_acc"]:.4f} | F1={r_s3["macro_f1"]:.4f} | AUC={r_s3["auc"]:.4f}')


MODEL: DenseNet121_S3_Hybrid
Frozen backbone | Head only | LR=0.001 | Epochs=30 | Early stop=7
Backbone frozen. Trainable params: 531,463

  Ep | Train Loss | Train Acc | Val Loss |  Val Acc |     Best | Pat
--------------------------------------------------------------------
   1 |     1.2297 |    0.2609 |   2.4271 |   0.1904 |   0.1904 | 0/7
   2 |     1.0131 |    0.3337 |   2.4465 |   0.1997 |   0.1997 | 0/7
   3 |     1.0117 |    0.2784 |   2.2921 |   0.2537 |   0.2537 | 0/7
   4 |     0.9545 |    0.3150 |   2.3807 |   0.1824 |   0.2537 | 1/7
   5 |     0.9298 |    0.3090 |   2.3629 |   0.2117 |   0.2537 | 2/7
   6 |     0.9127 |    0.3314 |   2.2832 |   0.2610 |   0.2610 | 0/7
   7 |     0.9407 |    0.3281 |   2.4196 |   0.1838 |   0.2610 | 1/7
   8 |     0.9462 |    0.3307 |   2.2279 |   0.2816 |   0.2816 | 0/7
   9 |     0.8934 |    0.3210 |   2.3121 |   0.2716 |   0.2816 | 1/7
  10 |     0.8518 |    0.3443 |   2.2184 |   0.2903 |   0.2903 | 0/7
  11 |     0.8786 |    0.3014 | 

## Step 15 — Final Comparison & Winner

In [25]:
all_results = [r_s1, r_s2, r_s3]

rows = [{'Strategy':      r['exp'],
         'Test Accuracy': round(r['test_acc'],  4),
         'Macro F1':      round(r['macro_f1'],  4),
         'AUC-ROC':       round(r['auc'],       4),
         'Best Val Acc':  round(r['best_val'],  4)}
        for r in all_results]

results_df = pd.DataFrame(rows).sort_values('Macro F1', ascending=False)
print('='*65)
print('DenseNet-121 — Strategy Comparison (ranked by Macro F1)')
print('='*65)
print(results_df.to_string(index=False))
results_df.to_csv(f'{RESULTS_DIR}/densenet121_strategy_comparison.csv', index=False)

winner = results_df.iloc[0]
print(f'\n🏆 BEST STRATEGY: {winner["Strategy"]}')
print(f'   Test Accuracy : {winner["Test Accuracy"]*100:.2f}%')
print(f'   Macro F1      : {winner["Macro F1"]:.4f}')
print(f'   AUC-ROC       : {winner["AUC-ROC"]:.4f}')
print(f'\n-> Use this strategy for DenseNet-121 contribution experiments.')

DenseNet-121 — Strategy Comparison (ranked by Macro F1)
                       Strategy  Test Accuracy  Macro F1  AUC-ROC  Best Val Acc
 DenseNet121_S2_AugOversampling         0.6900    0.4991   0.9092        0.7137
DenseNet121_S1_WeightedLossOnly         0.4731    0.3671   0.8735        0.5027
          DenseNet121_S3_Hybrid         0.2575    0.2577   0.8617        0.2903

🏆 BEST STRATEGY: DenseNet121_S2_AugOversampling
   Test Accuracy : 69.00%
   Macro F1      : 0.4991
   AUC-ROC       : 0.9092

-> Use this strategy for DenseNet-121 contribution experiments.


In [26]:
# ── All 3 learning curves overlaid ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#2196F3', '#FF9800', '#4CAF50']
labels = ['S1-WeightedLoss', 'S2-AugOversample', 'S3-Hybrid']

for r, color, label in zip(all_results, colors, labels):
    eps = range(1, len(r['history']['tr_loss']) + 1)
    axes[0].plot(eps, r['history']['vl_loss'], '-o', markersize=3, color=color, label=f'{label} Val')
    axes[0].plot(eps, r['history']['tr_loss'], '--',               color=color, alpha=0.4, label=f'{label} Train')
    axes[1].plot(eps, r['history']['vl_acc'],  '-o', markersize=3, color=color, label=f'{label} Val')
    axes[1].plot(eps, r['history']['tr_acc'],  '--',               color=color, alpha=0.4, label=f'{label} Train')

axes[0].set_title('All Strategies — Loss Curves', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)

axes[1].set_title('All Strategies — Accuracy Curves', fontsize=12)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)

plt.suptitle('DenseNet-121 — Strategy Comparison: Learning Curves', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/densenet121_all_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

Saved.


In [27]:
# ── Bar chart comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
metrics = ['Test Accuracy', 'Macro F1', 'AUC-ROC']
colors  = ['#2196F3', '#FF9800', '#4CAF50']

for ax, metric, color in zip(axes, metrics, colors):
    sd   = results_df.sort_values(metric, ascending=True)
    bars = ax.barh(sd['Strategy'], sd[metric], color=color, alpha=0.85)
    for bar, val in zip(bars, sd[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel(metric)
    ax.set_title(f'Ranked by {metric}')
    ax.set_xlim(0, 1.05)
    ax.grid(True, alpha=0.2, axis='x')

plt.suptitle('DenseNet-121 — Balancing Strategy Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/densenet121_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', f'{RESULTS_DIR}/densenet121_strategy_comparison.png')

Saved to /workspace/results/densenet121_strategy_comparison.png
